# NLP Project 2 – Whisper ASR Evaluation on Swiss German

**Dataset:** STT4SG-350 (Swiss German, 7 Dialektregionen)  
**Forschungsfrage:** Wie verändert sich die ASR-Qualität von Whisper v1→v2→v3 auf Schweizerdeutsch, und welche Fehlertypen dominieren?  
**Modelle:** `openai/whisper-large-v1`, `openai/whisper-large-v2`, `openai/whisper-large-v3`  
**Metrik:** WER (Word Error Rate), optional CER  

**Authors:** Natalie Jakab  (jakabnat) & Sarruja Sabesan (sabessar)
**Kurs:** Introduction to NLP – FS 2026, ZHAW

---

## Notebook-Struktur

| Zelle | Was passiert |
|---|---|
| 0. Setup | Imports, Pfade, Hilfsfunktionen |
| 1. Test-Run (lokal) | 10 Samples, 1 Modell → schnell validieren |
| 2. Full-Run (Colab) | Alle Samples, alle 3 Modelle, Checkpoint alle 200 |
| 3. Ergebnisse laden & WER berechnen | CSV einlesen, WER overall + per Region |
| 4. Visualisierung | Grouped Bar Chart WER per Modell/Region |

---
## 0. Setup

In [ ]:
# ── Installations (nur beim ersten Mal / auf Colab nötig) ──────────────────
!pip install torch transformers datasets jiwer soundfile librosa ffmpeg-python --quiet
!apt-get install ffmpeg -y   # ffmpeg (externes Tool) muss separat installiert werden

# ffmpeg (externes Tool) muss separat installiert werden (in PowerShell/Anaconda Prompt):
# conda install -c conda-forge ffmpeg -y 

# evtl. Kernel neustarten wenns import nicht direkt funktionert



In [2]:
import os
import csv
import time
import torch
import librosa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import transformers
from transformers import pipeline #für Whisper
from jiwer import wer, cer

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
print(f'Tranformer version: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')

DEVICE = 0 if torch.cuda.is_available() else -1
print(f'Using device:    {"GPU" if DEVICE == 0 else "CPU"}')

PyTorch version: 2.11.0+cpu
CUDA available:  False
Tranformer version: 5.4.0
Using device:    CPU


In [3]:
# ── Pfade konfigurieren ────────────────────────────────────────────────────
# Lokal:
DATA_DIR    = 'data/'               # Ordner mit test.tsv und clips__test/
AUDIO_DIR   = 'data/clips__test/'   # MP3-Files
RESULTS_DIR = 'results/'
TEST_FILE   = 'data/test.tsv'       # TSV-File mit Ground Truth

# Auf Colab (Google Drive): diese Zeilen stattdessen einkommentieren:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/NLP_Project2/data/'
# AUDIO_DIR   = '/content/drive/MyDrive/NLP_Project2/data/clips__test/'
# RESULTS_DIR = '/content/drive/MyDrive/NLP_Project2/results/'
# TEST_FILE   = '/content/drive/MyDrive/NLP_Project2/data/test.tsv'

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Data dir:    {DATA_DIR}')
print(f'Audio dir:   {AUDIO_DIR}')
print(f'Results dir: {RESULTS_DIR}')
print(f'Test file:   {TEST_FILE}')

Data dir:    data/
Audio dir:   data/clips__test/
Results dir: results/
Test file:   data/test.tsv


In [4]:
df_test = pd.read_csv(TEST_FILE, sep='\t')
print(f'Test set: {len(df_test)} Samples')
print(f'Columns:  {df_test.columns.tolist()}')
df_test.head(3)

Test set: 24605 Samples
Columns:  ['path', 'duration', 'sentence', 'sentence_source', 'client_id', 'dialect_region', 'canton', 'zipcode', 'age', 'gender']


,path,duration,sentence,sentence_source,client_id,dialect_region,canton,zipcode,age,gender
0,8800b4de-fa22-4fdc-9f29-457c4010fd57/d57790928...,4.478667,"Die Steuerfrage wäre eine andere, die Planung ...",parliament,8800b4de-fa22-4fdc-9f29-457c4010fd57,Basel,BS,4001,fourties,female
1,887b50f8-215b-4a1d-8f32-13516da6506f/73ca6c9b0...,6.134667,"Ich bin der Meinung, dass ihr noch nicht alle ...",parliament,887b50f8-215b-4a1d-8f32-13516da6506f,Bern,BE,3270,sixties,female
2,31cab952-98eb-45cb-a243-c951519c5c40/3aca44313...,5.126667,Dann änderte sie ihre Meinung und verlangte di...,news_switz,31cab952-98eb-45cb-a243-c951519c5c40,Innerschweiz,LU,6004,twenties,female


In [5]:
# ── Spalten prüfen & anpassen ──────────────────────────────────────────────
# STT4SG-350 test.csv hat typischerweise diese Spalten:
#   path, sentence, locale (Dialektregion)
# Falls die Spaltennamen anders sind → hier anpassen:
PATH_COL   = 'path'       # Dateiname des Audioclips (z.B. "abc123.mp3")
REF_COL    = 'sentence'   # Ground-Truth Transkription (Hochdeutsch)
REGION_COL = 'canton'     # Dialektregion (z.B. "BE", "ZH", ...)

print('Dialektregionen:', df_test[REGION_COL].unique())
print('Verteilung:')
print(df_test[REGION_COL].value_counts())

Dialektregionen: ['BS' 'BE' 'LU' 'SO' 'SG' 'ZG' 'VS' 'AG' 'TG' 'BL' 'ZH' 'GR' 'SH' 'UR'
 'TI' 'GL']
Verteilung:
canton
VS    3515
GR    3159
ZH    2532
BE    2494
SG    2450
BS    2261
LU    1704
AG    1550
BL    1254
SO     813
ZG     723
TG     721
UR     365
GL     364
TI     356
SH     344
Name: count, dtype: int64


---
## 1. Test-Run (lokal) – 10 Samples, 1 Modell

Zuerst lokal mit wenigen Samples validieren, dass die Pipeline funktioniert.  
Wenn das klappt → auf Colab mit dem Full-Run.

In [6]:
# ── Hilfsfunktion: Whisper-Pipeline laden ─────────────────────────────────
def load_whisper(model_id):
    """Lädt Whisper als HuggingFace pipeline."""
    print(f'Lade Modell: {model_id} ...')
    asr = pipeline(
        task='automatic-speech-recognition',
        model=model_id,
        device=DEVICE,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    # Sprache auf Deutsch zwingen (wichtig für Schweizerdeutsch!)
    asr.model.config.forced_decoder_ids = (
        asr.tokenizer.get_decoder_prompt_ids(language='german', task='transcribe')
    )
    print(f'Modell geladen: {model_id}')
    return asr


# ── Hilfsfunktion: Audio-Pfad auflösen ────────────────────────────────────
def resolve_audio_path(row):
    """Gibt den vollen Pfad zum Audioclip zurück."""
    return os.path.join(AUDIO_DIR, row[PATH_COL])


# ── Hilfsfunktion: Einen Clip transkribieren ───────────────────────────────
def transcribe_clip(asr_pipeline, audio_path):
    """Transkribiert einen Audioclip und gibt den Text zurück."""
    result = asr_pipeline(audio_path)
    return result['text'].strip()


print('Hilfsfunktionen definiert.')

Hilfsfunktionen definiert.


In [ ]:
# ── Test-Run: 10 Samples mit whisper-large-v2 ─────────────────────────────
N_TEST = 10
TEST_MODEL = 'openai/whisper-large-v2'

df_sample = df_test.head(N_TEST).copy()

asr = load_whisper(TEST_MODEL)

predictions = []
for i, row in df_sample.iterrows():
    audio_path = resolve_audio_path(row)
    if not os.path.exists(audio_path):
        print(f'  ⚠️  Datei nicht gefunden: {audio_path}')
        predictions.append('')
        continue
    hyp = transcribe_clip(asr, audio_path)
    predictions.append(hyp)
    print(f'  [{i+1}/{N_TEST}] REF: {row[REF_COL]}')
    print(f'         HYP: {hyp}')
    print()

df_sample['hypothesis'] = predictions

# WER berechnen
refs = df_sample[REF_COL].str.lower().tolist()
hyps = df_sample['hypothesis'].str.lower().tolist()
test_wer = wer(refs, hyps)
print(f'Test-Run WER ({N_TEST} Samples, {TEST_MODEL}): {test_wer:.4f}')

Lade Modell: openai/whisper-large-v2 ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Modell geladen: openai/whisper-large-v2


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

  [1/10] REF: Die Steuerfrage wäre eine andere, die Planung eine nachhaltigere.
          HYP: Die Steuerfrage wäre eine andere, die Planung eine nachhaltigere.

  [2/10] REF: Ich bin der Meinung, dass ihr noch nicht alle Sparmöglichkeiten ausgeschöpft habt.
          HYP: Ich bin der Meinung, dass er noch nicht alle Sparmöglichkeiten ausgeschöpft hat.

  [3/10] REF: Dann änderte sie ihre Meinung und verlangte die Tausender zurück.
          HYP: Dann hat sie ihre Meinung geändert und den Tausender zurückverlangt.

  [4/10] REF: Und neue Türen haben sich nicht geöffnet.
          HYP: Und neue Türen haben sich nicht geöffnet.

  [5/10] REF: Der Gemeinderat nimmt es auch als Postulat entgegen.
          HYP: Der Gemeinderod nimmt sie auch als Post und Art entgegen.

  [6/10] REF: Der Syrer soll Teil dieser Gruppe gewesen sein.
          HYP: Der Säurer soll ein Teil dieser Gruppe gewesen sein.

  [7/10] REF: Land ist ein knappes Gut und nicht vermehrbar.
          HYP: Land ist ein knap

---
## 2. Full-Run (Colab) – Alle Samples, alle 3 Modelle

⚠️ **Dieser Block ist für Colab mit GPU (T4).**  
Laufzeit ca. 3–6h total (alle 3 Modelle).  
Checkpoints werden alle 200 Samples gespeichert → bei Absturz kann man weitermachen.

**Wie starten:**
1. Google Drive mounten (Zeilen im Setup oben einkommentieren)
2. Alle Zellen von oben nach unten ausführen
3. Dann diese Zelle starten

In [ ]:
# ── Konfig Full-Run ────────────────────────────────────────────────────────
MODELS = [
    'openai/whisper-large-v1',
    'openai/whisper-large-v2',
    'openai/whisper-large-v3',
]
CHECKPOINT_EVERY = 200  # alle 200 Samples speichern


def get_checkpoint_path(model_id):
    """Gibt den Checkpoint-Pfad für ein Modell zurück."""
    safe_name = model_id.replace('/', '_')
    return os.path.join(RESULTS_DIR, f'checkpoint_{safe_name}.csv')


def load_checkpoint(model_id):
    """Lädt bereits fertige Samples aus dem Checkpoint (falls vorhanden)."""
    path = get_checkpoint_path(model_id)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'  Checkpoint gefunden: {len(df)} Samples bereits verarbeitet.')
        return df
    return pd.DataFrame()


def save_checkpoint(rows, model_id):
    """Speichert aktuelle Ergebnisse als Checkpoint-CSV."""
    path = get_checkpoint_path(model_id)
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)


print('Full-Run Konfig bereit.')
print(f'Modelle: {MODELS}')
print(f'Checkpoint alle {CHECKPOINT_EVERY} Samples → {RESULTS_DIR}')

In [ ]:
# ── Full-Run Loop ──────────────────────────────────────────────────────────
for model_id in MODELS:
    print(f'\n{"="*60}')
    print(f'Modell: {model_id}')
    print(f'{"="*60}')

    # Checkpoint laden (falls schon Fortschritt vorhanden)
    df_done = load_checkpoint(model_id)
    done_paths = set(df_done[PATH_COL].tolist()) if len(df_done) > 0 else set()

    # Noch ausstehende Samples
    df_todo = df_test[~df_test[PATH_COL].isin(done_paths)].reset_index(drop=True)
    print(f'Noch zu verarbeiten: {len(df_todo)} / {len(df_test)} Samples')

    if len(df_todo) == 0:
        print('  → Bereits fertig, überspringe.')
        continue

    # Modell laden
    asr = load_whisper(model_id)

    rows = df_done.to_dict('records')  # bisherige Ergebnisse
    t_start = time.time()

    for i, row in df_todo.iterrows():
        audio_path = resolve_audio_path(row)

        if not os.path.exists(audio_path):
            hyp = ''
        else:
            try:
                hyp = transcribe_clip(asr, audio_path)
            except Exception as e:
                print(f'  ⚠️  Fehler bei {audio_path}: {e}')
                hyp = ''

        rows.append({
            PATH_COL:     row[PATH_COL],
            REF_COL:      row[REF_COL],
            REGION_COL:   row[REGION_COL],
            'model':      model_id,
            'hypothesis': hyp,
        })

        # Checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(rows, model_id)
            elapsed = time.time() - t_start
            print(f'  Checkpoint: {i+1}/{len(df_todo)} Samples | {elapsed/60:.1f} min')

    # Finales Speichern
    save_checkpoint(rows, model_id)
    print(f'  ✅ Fertig: {len(rows)} Samples gespeichert → {get_checkpoint_path(model_id)}')

    # Modell aus GPU-Speicher entfernen
    del asr
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print('\n✅ Full-Run abgeschlossen!')

---
## 3. Ergebnisse laden & WER berechnen

Nach dem Full-Run: CSVs einlesen, WER overall und per Dialektregion.

In [ ]:
# ── Alle Checkpoint-CSVs laden und zusammenführen ─────────────────────────
all_dfs = []
for model_id in MODELS:
    path = get_checkpoint_path(model_id)
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['model'] = model_id  # sicherstellen
        all_dfs.append(df)
        print(f'Geladen: {model_id} → {len(df)} Samples')
    else:
        print(f'⚠️  Kein Checkpoint für {model_id}')

df_all = pd.concat(all_dfs, ignore_index=True)
print(f'\nTotal: {len(df_all)} Zeilen')
df_all.head()

In [ ]:
# ── WER-Berechnung: Overall pro Modell ────────────────────────────────────
def compute_wer_for_group(df_group):
    refs = df_group[REF_COL].fillna('').str.lower().tolist()
    hyps = df_group['hypothesis'].fillna('').str.lower().tolist()
    return wer(refs, hyps)


print('=== WER Overall (pro Modell) ===')
wer_overall = {}
for model_id in MODELS:
    df_m = df_all[df_all['model'] == model_id]
    w = compute_wer_for_group(df_m)
    wer_overall[model_id] = w
    label = model_id.split('/')[-1]  # z.B. "whisper-large-v2"
    print(f'  {label}: WER = {w:.4f} ({w*100:.2f}%)')

In [ ]:
# ── WER-Berechnung: Pro Dialektregion × Modell ─────────────────────────────
regions = sorted(df_all[REGION_COL].unique())
models_short = [m.split('/')[-1] for m in MODELS]  # z.B. ["whisper-large-v1", ...]

wer_by_region = {}
for model_id, label in zip(MODELS, models_short):
    wer_by_region[label] = {}
    df_m = df_all[df_all['model'] == model_id]
    for region in regions:
        df_r = df_m[df_m[REGION_COL] == region]
        if len(df_r) == 0:
            wer_by_region[label][region] = float('nan')
        else:
            wer_by_region[label][region] = compute_wer_for_group(df_r)

df_wer = pd.DataFrame(wer_by_region).T  # rows = Modelle, cols = Regionen
print('=== WER pro Dialektregion ===')
print(df_wer.round(4))

In [ ]:
# ── Ergebnisse als CSV speichern ──────────────────────────────────────────
os.makedirs(RESULTS_DIR, exist_ok=True)
df_wer.to_csv(os.path.join(RESULTS_DIR, 'wer_by_region.csv'))

df_overall = pd.DataFrame([
    {'model': m.split('/')[-1], 'wer_overall': w}
    for m, w in wer_overall.items()
])
df_overall.to_csv(os.path.join(RESULTS_DIR, 'wer_overall.csv'), index=False)
print('Gespeichert: wer_by_region.csv, wer_overall.csv')

---
## 4. Visualisierung – Grouped Bar Chart WER pro Modell & Region

In [ ]:
# ── Grouped Bar Chart ─────────────────────────────────────────────────────
os.makedirs('figures/', exist_ok=True)

x = np.arange(len(regions))
n_models = len(models_short)
bar_width = 0.25
colors = ['#4C72B0', '#DD8452', '#55A868']  # blau, orange, grün

fig, ax = plt.subplots(figsize=(12, 5))

for j, (label, color) in enumerate(zip(models_short, colors)):
    vals = [wer_by_region[label].get(r, float('nan')) for r in regions]
    offset = (j - n_models / 2 + 0.5) * bar_width
    bars = ax.bar(x + offset, vals, width=bar_width, label=label, color=color)
    # Werte über den Balken
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Dialektregion')
ax.set_ylabel('WER')
ax.set_title('WER pro Dialektregion und Whisper-Modell')
ax.set_xticks(x)
ax.set_xticklabels(regions)
ax.legend()
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/fig_wer_by_region.png', dpi=150)
plt.show()
print('Gespeichert: figures/fig_wer_by_region.png')

In [ ]:
# ── Overall WER Balkendiagramm (einfach) ──────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
vals = [wer_overall[m] for m in MODELS]
bars = ax.bar(models_short, vals, color=colors[:len(MODELS)])
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{v:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('WER (overall)')
ax.set_title('Overall WER pro Whisper-Modell')
ax.set_ylim(0, max(vals) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig_wer_overall.png', dpi=150)
plt.show()
print('Gespeichert: figures/fig_wer_overall.png')

---
## 5. (Optional) CER berechnen

In [ ]:
# ── CER Overall pro Modell ────────────────────────────────────────────────
print('=== CER Overall (pro Modell) ===')
for model_id in MODELS:
    df_m = df_all[df_all['model'] == model_id]
    refs = df_m[REF_COL].fillna('').str.lower().tolist()
    hyps = df_m['hypothesis'].fillna('').str.lower().tolist()
    c = cer(refs, hyps)
    label = model_id.split('/')[-1]
    print(f'  {label}: CER = {c:.4f} ({c*100:.2f}%)')